# Application Level Profiling

We follow a *top-down approach* to performance analysis, starting with a *whole application* performance overview and then narrowing down to specific *hot spots*.
An initial overview can be obtained using Nsight Systems, using either solely the command line interface, or by complementing the analysis with the provided GUI.

## Nsight Systems CLI

First, we compile and execute our benchmark application to make sure that results are as expected.

In [ ]:
!nvc++ -O3 -march=native -std=c++17 -mp=gpu -target=gpu ../src/stencil-2d/stencil-2d-omp-target-v0.cpp -o ../build/stencil-2d-omp-target-v0
!../build/stencil-2d-omp-target-v0 double 8192 8192 2 256

Next, we profile our binary with `nsys profile`.
Further command line arguments are:
* `--stats=true`: prints a summary of performance statistics on the command line
* `-o ...`: sets the target output profile file
* `--force-overwrite=true`: replaces the profile file if it already exists (instead of aborting)

In [ ]:
!nsys profile --stats=true -o ../profiles/stencil-2d-omp-target-v0 --force-overwrite=true ../build/stencil-2d-omp-target-v0 double 8192 8192 2 256

The output of the command line is organized in multiple categories.
A possible output for an Nvidia A40 is copied in below:

### Possible Output

```bash
[4/8] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls     Avg (ns)         Med (ns)        Min (ns)       Max (ns)     StdDev (ns)            Name         
 --------  ---------------  ---------  ---------------  ---------------  -------------  -------------  ------------  ----------------------
     74.5  117,237,417,968      4,248     27,598,262.2     10,098,956.0          2,164    100,327,529  35,654,384.2  poll                  
     25.4   40,001,058,300         10  4,000,105,830.0  4,000,110,931.0  4,000,055,861  4,000,129,290      21,478.7  pthread_cond_timedwait
      0.1      122,325,955        579        211,271.1         33,644.0          1,042     22,464,342   1,069,187.9  ioctl                 
      0.0        2,207,871         25         88,314.8         12,123.0          7,765      1,429,593     282,192.9  mmap64                
      0.0        1,758,990         10        175,899.0         79,501.0         22,583        703,255     246,959.0  sem_timedwait         
      0.0          777,807          3        259,269.0        275,112.0        218,535        284,160      35,565.6  pthread_create        
      0.0          755,984         20         37,799.2         11,071.0          5,150        378,909      82,771.6  mmap                  
      0.0          303,148         33          9,186.3          8,286.0          1,303         36,550       5,963.6  fclose                
      0.0          273,530         57          4,798.8          4,458.0          1,964         13,175       2,386.7  open64                
      0.0          271,332         47          5,773.0          3,957.0          1,493         31,991       5,786.9  fopen                 
      0.0          244,574          1        244,574.0        244,574.0        244,574        244,574           0.0  pthread_cond_wait     
      0.0           97,006          9         10,778.4          7,645.0          4,328         36,720      10,465.5  munmap                
      0.0           80,503          1         80,503.0         80,503.0         80,503         80,503           0.0  fgets                 
      0.0           39,193          6          6,532.2          5,349.5          2,204         16,822       5,242.7  open                  
      0.0           28,885         14          2,063.2          2,329.5          1,042          4,569         945.0  read                  
      0.0           27,633         10          2,763.3          2,800.0          2,124          3,416         432.9  write                 
      0.0           26,430          4          6,607.5          6,597.5          2,995         10,240       3,708.8  pipe2                 
      0.0           24,507          7          3,501.0          2,204.0          1,152          7,304       2,448.3  close                 
      0.0           24,245          5          4,849.0          3,286.0          1,032         11,521       4,506.2  fwrite                
      0.0           15,970          2          7,985.0          7,985.0          3,857         12,113       5,837.9  socket                
      0.0           13,315          1         13,315.0         13,315.0         13,315         13,315           0.0  connect               
      0.0            5,691          1          5,691.0          5,691.0          5,691          5,691           0.0  pthread_cond_broadcast
      0.0            2,856          1          2,856.0          2,856.0          2,856          2,856           0.0  putc                  
      0.0            2,805          2          1,402.5          1,402.5          1,252          1,553         212.8  fcntl                 
      0.0            2,014          1          2,014.0          2,014.0          2,014          2,014           0.0  bind                  
      0.0            1,724          1          1,724.0          1,724.0          1,724          1,724           0.0  sigaction             
```

```bash
[5/8] Executing 'cuda_api_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)          Name        
 --------  ---------------  ---------  ------------  ------------  ----------  ----------  -----------  --------------------
     53.9   22,268,529,271        516  43,156,064.5  43,709,611.0  39,766,668  47,241,023  3,200,509.1  cuMemcpyDtoHAsync_v2
     46.0   19,014,806,194        516  36,850,399.6  36,803,284.0  36,609,547  38,686,869    208,369.8  cuMemcpyHtoDAsync_v2
      0.0        5,902,946        258      22,879.6      21,811.5      14,307      57,619      5,759.9  cuLaunchKernel      
      0.0        3,165,558          5     633,111.6     187,967.0      52,530   2,422,337  1,007,605.0  cuMemFree_v2        
      0.0        1,337,458          1   1,337,458.0   1,337,458.0   1,337,458   1,337,458          0.0  cuMemAllocHost_v2   
      0.0          881,302          4     220,325.5     204,287.5       7,594     465,133    187,840.2  cuMemAlloc_v2       
      0.0          642,897        258       2,491.8       2,354.5       1,613      11,351      1,034.1  cuStreamSynchronize 
      0.0          593,968          1     593,968.0     593,968.0     593,968     593,968          0.0  cuMemFreeHost       
      0.0          518,153          1     518,153.0     518,153.0     518,153     518,153          0.0  cuModuleLoadDataEx  
      0.0          166,025          1     166,025.0     166,025.0     166,025     166,025          0.0  cuMemAllocManaged   
      0.0            4,639          5         927.8         681.0         200       2,324        877.8  cuCtxSetCurrent     
      0.0            1,463          1       1,463.0       1,463.0       1,463       1,463          0.0  cuInit              
```

```bash
[6/8] Executing 'cuda_gpu_kern_sum' stats report

 Time (%)  Total Time (ns)  Instances  Avg (ns)   Med (ns)   Min (ns)  Max (ns)  StdDev (ns)                     Name
 --------  ---------------  ---------  ---------  ---------  --------  --------  -----------  -------------------------------------------
    100.0    1,426,755,432        258  5,530,059.8  5,531,510.0  5,504,786  5,550,996     10,551.0  nvkernel__Z9stencil2dIdEvPKT_PS0_mm_F1L5_6
```

```bash
[7/8] Executing 'cuda_gpu_mem_time_sum' stats report

 Time (%)  Total Time (ns)  Count    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)           Operation          
 --------  ---------------  -----  ------------  ------------  ----------  ----------  -----------  ----------------------------
     52.3   20,795,826,190    516  40,301,988.7  40,265,268.5  39,685,863  41,723,845    484,363.8  [CUDA memcpy Device-to-Host]
     47.7   19,001,128,255    516  36,823,892.0  36,778,375.0  36,578,982  38,600,744    203,820.0  [CUDA memcpy Host-to-Device]
```


```bash
[8/8] Executing 'cuda_gpu_mem_size_sum' stats report

 Total (MB)   Count  Avg (MB)  Med (MB)  Min (MB)  Max (MB)  StdDev (MB)           Operation          
 -----------  -----  --------  --------  --------  --------  -----------  ----------------------------
 277,025.391    516   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Device-to-Host]
 277,025.391    516   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Host-to-Device]
```

## Exercise - Interpret System Command Line Output

Look at the performance data and try to answer the following questions:
* What is the largest execution time contributor?
* How often is the main kernel launched?
* Is synchronization applied after every kernel launch?
* Does the time spent in synchronization match your expectation?
* Do transfer sizes match your expectation?

### Possible Solution

Looking at the statistics we can see multiple effects:
* The number of kernel instances matches our expectation ($2 + 256$).
* The number of memory transfers seems to be related to the number of kernel instances.
* Comparing the time spent in GPU synchronization (`cuStreamSynchronize`) and kernel execution time shows a mismatch.
* Comparing aggregated memory transfer and kernel execution times reveals an order of magnitude in difference.
  * \> Even if the kernel could be accelerated, overall performance will most likely not increase.
  * \> These numbers could be used to approximate a minimum number of iterations at which the memory transfers get amortized (assuming that the transfer times *don't scale with the number of iterations*).
* The size per transfer matches our expectation (8192**2 $\cdot$ 8 B $\approx$ 537 MB).

## Nsight Systems GUI

Next, we further investigate the problematic memory transfers by opening up the generated `stencil-2d-omp-target-v0` report file which is in the `../profiles` folder.

## Exercise - Visualize Timeline

Download the produced file and open it with your local installation of Nsight Systems.
Can you connect the timeline to some of the effects seen previously?

### Possible Solution

As maybe already suspected, the timeline shows a recurring pattern of
* two memory transfers (HtoD),
* a kernel call, and
* two memory transfers (DtoH)

Additionally, `cudaStreamSynchronize` is only called at the end of this pattern, which partly explains the deviation from the kernel execution time.

You might also be wondering why the asynchronous data transfers show up as synchronous.
This is an artefact of the way the host memory was allocated which, by default, gives **pageable** memory.
**Pinned memory** (or page-locked memory), in contrast, is allocated such that it cannot be paged to disk.

Without pinned memory, the CUDA runtime is required to stage memory transfers via a pinned buffer which has two effects:
* transfers are done synchronously and
* transfer rates are lower than expected (compare the achieved $\thicksim 13 \text{GB/s}$ with the theoretical maximum of $31.5 \text{GB/s}$).

Pinned memory can be allocated via specialized allocators, e.g. `cudaMallocHost`.

## Stencil Code Optimization 1 - Reduce Data Transfers

Having pinpointed our performance bug, we can now optimize data transfers in our application.
One straight-forward way is adding unstructured data primitives in our code, basically spanning a region at whose begin and end data is copied *only one time*.
The updated version is available at [stencil-2d-omp-target-v1.cpp](../src/stencil-2d/stencil-2d-omp-target-v1.cpp), and can be compiled, executed and profiled using the following cells.

In [ ]:
!nvc++ -O3 -march=native -std=c++17 -mp=gpu -target=gpu ../src/stencil-2d/stencil-2d-omp-target-v1.cpp -o ../build/stencil-2d-omp-target-v1

In [ ]:
!../build/stencil-2d-omp-target-v1 double 8192 8192 2 256

In [ ]:
!nsys profile --stats=true -o ../profiles/stencil-2d-omp-target-v1 --force-overwrite=true ../build/stencil-2d-omp-target-v1 double 8192 8192 2 256

### Possible Output

```bash
[4/8] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)    Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  ------------  --------  -----------  ------------  ----------------------
     97.6    4,810,739,901        193  24,926,113.5  10,097,793.0     1,934  100,163,868  33,833,488.0  poll                  
      2.3      111,634,312        569     196,193.9      32,723.0     1,062   22,263,762   1,025,360.2  ioctl                 
      0.0        2,154,801         25      86,192.0      12,393.0     7,665    1,415,667     279,429.6  mmap64                
      0.0        2,128,760         10     212,876.0      82,171.0    30,728      809,817     295,080.8  sem_timedwait         
      0.0          842,228          3     280,742.7     281,995.0   229,355      330,878      50,773.1  pthread_create        
      0.0          737,923         20      36,896.2      12,709.5     5,040      360,274      78,907.4  mmap                  
      0.0          301,926         34       8,880.2       8,466.0     1,042       36,529       5,901.0  fclose                
      0.0          279,463         47       5,946.0       3,918.0     1,513       32,702       5,934.0  fopen                 
      0.0          263,687         57       4,626.1       4,428.0     1,974       14,848       2,220.9  open64                
      0.0          213,475          1     213,475.0     213,475.0   213,475      213,475           0.0  pthread_cond_wait     
      0.0           95,160          9      10,573.3       6,392.0     3,467       36,610      10,765.5  munmap                
      0.0           81,514          1      81,514.0      81,514.0    81,514       81,514           0.0  fgets                 
      0.0           33,202          6       5,533.7       5,189.5     2,114       10,780       2,976.4  open                  
      0.0           30,727          4       7,681.8       7,834.5     4,188       10,870       3,389.5  pipe2                 
      0.0           29,646         14       2,117.6       2,309.5     1,092        4,258         855.0  read                  
      0.0           26,671         10       2,667.1       2,625.0     1,823        4,419         719.8  write                 
      0.0           22,953          8       2,869.1       1,979.0     1,173        6,372       1,932.0  close                 
      0.0           21,661          5       4,332.2       3,627.0     1,052        7,153       2,726.9  fwrite                
      0.0           16,772          2       8,386.0       8,386.0     4,078       12,694       6,092.4  socket                
      0.0           15,129          1      15,129.0      15,129.0    15,129       15,129           0.0  connect               
      0.0           11,522          1      11,522.0      11,522.0    11,522       11,522           0.0  putc                  
      0.0            5,901          1       5,901.0       5,901.0     5,901        5,901           0.0  pthread_cond_broadcast
      0.0            4,479          3       1,493.0       1,283.0     1,142        2,054         490.9  fcntl                 
      0.0            2,034          1       2,034.0       2,034.0     2,034        2,034           0.0  bind                  
      0.0            1,914          1       1,914.0       1,914.0     1,914        1,914           0.0  sigaction             
      0.0            1,032          1       1,032.0       1,032.0     1,032        1,032           0.0  fflush                
```

```bash
[5/8] Executing 'cuda_api_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)          Name        
 --------  ---------------  ---------  ------------  ------------  ----------  ----------  -----------  --------------------
     89.5    1,429,493,983        260   5,498,053.8   5,538,425.5       1,603   5,580,471    485,022.9  cuStreamSynchronize 
      5.3       85,315,189          2  42,657,594.5  42,657,594.5  41,481,823  43,833,366  1,662,792.0  cuMemcpyDtoHAsync_v2
      4.7       75,058,233          2  37,529,116.5  37,529,116.5  36,887,524  38,170,709    907,348.8  cuMemcpyHtoDAsync_v2
      0.2        3,181,657          5     636,331.4     177,637.0      63,390   2,414,983  1,002,131.3  cuMemFree_v2        
      0.1        1,131,527          1   1,131,527.0   1,131,527.0   1,131,527   1,131,527          0.0  cuMemAllocHost_v2   
      0.1        1,055,401        258       4,090.7       3,717.0       3,296      25,659      2,020.6  cuLaunchKernel      
      0.1          895,200          4     223,800.0     198,321.5       8,527     490,030    199,282.6  cuMemAlloc_v2       
      0.0          695,670          1     695,670.0     695,670.0     695,670     695,670          0.0  cuModuleLoadDataEx  
      0.0          580,241          1     580,241.0     580,241.0     580,241     580,241          0.0  cuMemFreeHost       
      0.0          149,343          1     149,343.0     149,343.0     149,343     149,343          0.0  cuMemAllocManaged   
      0.0            7,143          5       1,428.6         461.0         191       4,248      1,736.1  cuCtxSetCurrent     
      0.0            1,512          1       1,512.0       1,512.0       1,512       1,512          0.0  cuInit              
```

```bash
[6/8] Executing 'cuda_gpu_kern_sum' stats report

 Time (%)  Total Time (ns)  Instances   Avg (ns)     Med (ns)    Min (ns)   Max (ns)   StdDev (ns)                     Name                   
 --------  ---------------  ---------  -----------  -----------  ---------  ---------  -----------  ------------------------------------------
    100.0    1,429,349,626        258  5,540,114.8  5,537,990.5  5,509,654  5,583,126     20,076.8  nvkernel__Z9stencil2dIdEvPKT_PS0_mm_F1L5_6
```

```bash
[7/8] Executing 'cuda_gpu_mem_time_sum' stats report

 Time (%)  Total Time (ns)  Count    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)           Operation          
 --------  ---------------  -----  ------------  ------------  ----------  ----------  -----------  ----------------------------
     53.0       84,530,360      2  42,265,180.0  42,265,180.0  40,819,550  43,710,810  2,044,429.6  [CUDA memcpy Device-to-Host]
     47.0       74,966,496      2  37,483,248.0  37,483,248.0  36,855,713  38,110,783    887,468.5  [CUDA memcpy Host-to-Device]
```

```bash
[8/8] Executing 'cuda_gpu_mem_size_sum' stats report

 Total (MB)  Count  Avg (MB)  Med (MB)  Min (MB)  Max (MB)  StdDev (MB)           Operation          
 ----------  -----  --------  --------  --------  --------  -----------  ----------------------------
  1,073.742      2   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Device-to-Host]
  1,073.742      2   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Host-to-Device]
```

## Exercise - Compare Systems Output to Base Version

Revisit the observations from before.
Which of them still hold, which are now different?

Also visualize the time line - are the code changes reflected as expected?

### Possible Solution

Looking at the statistics we can now compare the effects previously discussed:
* The number of kernel instances matches our expectation ($2 + 256$).
* ~~The number of memory transfers seems to be related to the number of kernel instances.~~
* ~~Comparing the time spent in GPU synchronization (`cuStreamSynchronize`) and kernel execution time shows a mismatch.~~
* ~~Comparing aggregated memory transfer and kernel execution times reveals an order of magnitude in difference.~~
* The size per transfer matches our expectation ($8192 \cdot 8192 \cdot 8 \text{B} \approx 537 \text{MB}$).

Opening up the generated profile output in our GUI reveals what we already expected: single staging parts at the beginning and the end of our application with multiple kernel calls in between.
The output file is once again collected in the `../profiles` folder.

## NVTX Marker Extensions

Our application is quite straight-forward and the complexity of our analysis so far was manageable.
For more complex codes, however, the analysis can become more evolved and, in this case, marking code regions can help.
We will use the **NVidia Tools eXtensions library (NVTX)** for this task.
The project is [open source](https://github.com/NVIDIA/NVTX) and well [documented](https://nvidia.github.io/NVTX/doxygen-cpp/).

The Nvidia HPC SDK (NVHPC) already includes NVTX.
Depending on the features required, this might already be sufficient.
In case a developer version is required, the latest version can be obtained from the GitHub repository using
```
git clone https://github.com/NVIDIA/NVTX.git
```
Since it is a **header-only library** no additional steps are necessary apart from specifying the include path during compilation with
```
-I/path/to/nvtx/include
```

After preparation, the next steps are adding the necessary header and modifying the code.

```c++
#include <nvtx3/nvtx3.hpp>
```

The easiest way to get started is pushing and popping NVTX ranges using the C API:

```c++
    nvtxRangePushA("important work");

    // meaningful work

    nvtxRangePop(); // important work
```

The C++ API additionally exposes ranges for different scopes, the most flexible being the `scoped_range`:

```c++
    { // stand-alone scope, loop body, function body, ...
        nvtx3::scoped_range loop{"main loop"};

        // meaningful work
    } // no explicit pop
```

One specialization is the function range macro which takes the surrounding functions name as the constructed range's name:

```c++
void seriousWork(int someArg) {
    NVTX3_FUNC_RANGE(); // equivalent to nvtx3::scoped_range loop{"seriousWork"};

    // meaningful work
} // end of scoped range, no explicit pop
```

## Exercise - Annotate Stencil Example

Adapt the current version of our stencil application ([stencil-2d-omp-target-v1.cpp](../src/stencil-2d/stencil-2d-omp-target-v1.cpp)) with NVTX regions.
Implement the following extensions:
* Include the necessary header.
* Span regions around the target data enter and exit operations.
* Span a region inside the main work function (`stencil2d`).
* Span a region within and/ or around the timed iteration loop.
* Profile the annotated version and visualize the report file (`stencil-2d-omp-target-v1-nvtx.nsys-rep` in `../profiles`)

Use the following cells to compile, execute and profile.

In [ ]:
!nvc++ -O3 -march=native -std=c++17 -mp=gpu -target=gpu ../src/stencil-2d/stencil-2d-omp-target-v1.cpp -o ../build/stencil-2d-omp-target-v1

In [ ]:
!../build/stencil-2d-omp-target-v1 double 8192 8192 2 256

In [ ]:
!nsys profile --stats=true -o ../profiles/stencil-2d-omp-target-v1-nvtx --force-overwrite=true ../build/stencil-2d-omp-target-v1 double 8192 8192 2 8

### Possible Solution

[stencil-2d-omp-target-v1-nvtx.cpp](../src/stencil-2d/stencil-2d-omp-target-v1-nvtx.cpp) shows one way of using NVTX regions in action.
It can be compiled, executed and profiled using the following cells.

In [ ]:
!nvc++ -O3 -march=native -std=c++17 -mp=gpu -target=gpu ../src/stencil-2d/stencil-2d-omp-target-v1-nvtx.cpp -o ../build/stencil-2d-omp-target-v1-nvtx

In [ ]:
!../build/stencil-2d-omp-target-v1-nvtx double 8192 8192 2 256

In [ ]:
!nsys profile --stats=true -o ../profiles/stencil-2d-omp-target-v1-nvtx --force-overwrite=true ../build/stencil-2d-omp-target-v1-nvtx double 8192 8192 2 8

#### Possible Output

The command line output now features an additional section `nvtx_sum`

```bash
[3/8] Executing 'nvtx_sum' stats report

 Time (%)  Total Time (ns)  Instances    Avg (ns)       Med (ns)      Min (ns)     Max (ns)    StdDev (ns)   Style         Range      
 --------  ---------------  ---------  -------------  -------------  -----------  -----------  -----------  -------  -----------------
     54.2      301,684,523          1  301,684,523.0  301,684,523.0  301,684,523  301,684,523          0.0  PushPop  :enter data      
     19.6      108,793,454          1  108,793,454.0  108,793,454.0  108,793,454  108,793,454          0.0  PushPop  :exit data       
     10.2       56,998,935         10    5,699,893.5    5,558,041.5    5,527,157    7,036,913    470,057.9  PushPop  :stencil2d       
      8.0       44,403,385          1   44,403,385.0   44,403,385.0   44,403,385   44,403,385          0.0  PushPop  :timed iterations
      1.0        5,569,177          1    5,569,177.0    5,569,177.0    5,569,177    5,569,177          0.0  PushPop  :iteration3      
      1.0        5,566,362          1    5,566,362.0    5,566,362.0    5,566,362    5,566,362          0.0  PushPop  :iteration1      
      1.0        5,565,892          1    5,565,892.0    5,565,892.0    5,565,892    5,565,892          0.0  PushPop  :iteration7      
      1.0        5,551,424          1    5,551,424.0    5,551,424.0    5,551,424    5,551,424          0.0  PushPop  :iteration5      
      1.0        5,541,064          1    5,541,064.0    5,541,064.0    5,541,064    5,541,064          0.0  PushPop  :iteration2      
      1.0        5,538,790          1    5,538,790.0    5,538,790.0    5,538,790    5,538,790          0.0  PushPop  :iteration6      
      1.0        5,535,914          1    5,535,914.0    5,535,914.0    5,535,914    5,535,914          0.0  PushPop  :iteration0      
      1.0        5,527,679          1    5,527,679.0    5,527,679.0    5,527,679    5,527,679          0.0  PushPop  :iteration4      
```

As we can see, nested regions work without additional issues, but interpreting the first column (`Time (%)`) now needs to be done with additional care.

Downloading and opening the `stencil-2d-omp-target-v1-nvtx` report file, which is in the `../profiles` folder as usual, reveals an additional row in the timeline - `NVTX`.

## Next Step

The main execution time contributor is now the kernel execution.
Since we have only a single kernel, further hot spot analysis is not necessary in this case.
Instead, we directly focus on its performance in the [kernel level profiling](./04-kernel-level-profiling.ipynb) notebook.